# Лучшие параметры embedding по колонкам

Ноутбук читает полный результат перебора `uzal_embedding_results_all_columns.csv` и оставляет по одной лучшей строке на каждую колонку: минимальный `uzal_cost`, соответствующие `tau` и `dimension`.

In [17]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px

In [18]:
import subprocess

process = subprocess.Popen(
    ["julia", "--project=.", "main.jl"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, process.args)

Numeric columns: 117
[1/117] Lab1_G1_N1
[2/117] Lab1_G1_N2
[3/117] Lab1_G1_N3
[4/117] Lab1_G1_P2
[5/117] Lab1_G1_T4СЃСЂ
[6/117] Lab1_G1_T1
[7/117] Lab1_G1_T607
[8/117] Lab1_G1_T600
[9/117] Lab1_G1_T638
[10/117] Lab1_G1_T606
[11/117] Lab1_G1_T1002
[12/117] Lab1_G1_T1003
[13/117] Lab1_G2_Fc2
[14/117] Lab1_G2_F1
[15/117] Lab1_G2_Fc3
[16/117] Lab1_G2_2F1
[17/117] Lab1_G2_3F1
[18/117] Lab1_G2_FС‚Рє2
[19/117] Lab1_G2_FРЅС€
[20/117] Lab1_G2_FРјР°
[21/117] Lab1_G2_FРјРЅ
[22/117] Lab1_G2_FСЃС‚РІ
[23/117] Lab1_G2_Fc4
[24/117] Lab1_G2_FС†СЃ
[25/117] Lab1_G2_F2
[26/117] Lab1_G2_FРєРїР°
[27/117] Lab1_G2_2F2
[28/117] Lab1_G2_3F2
[29/117] Lab1_G2_FС‚Рє4
[30/117] Lab1_G2_3_77F2
[31/117] Lab1_G2_VoР“Р“
[32/117] Lab1_G2_Fc9
[33/117] Lab1_G2_FСЃ8
[34/117] Lab1_G2_F3
[35/117] Lab1_G2_2F3
[36/117] Lab1_G2_3F3
[37/117] Lab1_G2_FС‚Рє9
[38/117] Lab1_G2_FС‚Рє8
[39/117] Lab1_G2_FРЅ9
[40/117] Lab1_G2_FРІ9
[41/117] Lab1_G2_VoРЎРў
[42/117] Lab1_G3_N3
[43/117] Lab1_G3_Lm
[44/117] Lab1_G3_dPf1
[45/117] Lab1_G3_Pm
[4

In [19]:
RESULTS_PATH = Path("uzal_embedding_results_all_columns.csv")
SKIPPED_PATH = Path("uzal_embedding_skipped_columns.csv")
BEST_OUTPUT_PATH = Path("uzal_best_choices_by_column.csv")

In [20]:
results = pd.read_csv(RESULTS_PATH)
results.head(), results.shape

(       column  tau  dimension  uzal_cost
 0  Lab1_G1_N1   98          4   1.028918
 1  Lab1_G1_N1   73          5   1.029342
 2  Lab1_G1_N1   73          6   1.031967
 3  Lab1_G1_N1   74          5   1.032444
 4  Lab1_G1_N1   75          5   1.033032,
 (88000, 4))

In [21]:
required_columns = {"column", "tau", "dimension", "uzal_cost"}
missing_columns = required_columns - set(results.columns)
if missing_columns:
    raise ValueError(f"Missing columns in {RESULTS_PATH}: {sorted(missing_columns)}")

all_result_columns = set(results["column"].dropna())
finite_results = results.dropna(subset=["column", "tau", "dimension", "uzal_cost"]).copy()
finite_results["tau"] = finite_results["tau"].astype(int)
finite_results["dimension"] = finite_results["dimension"].astype(int)

print(f"Rows in full results: {len(results)}")
print(f"Columns in full results: {len(all_result_columns)}")
print(f"Rows with finite uzal_cost: {len(finite_results)}")
print(f"Columns with finite uzal_cost: {finite_results['column'].nunique()}")

Rows in full results: 88000
Columns in full results: 88
Rows with finite uzal_cost: 61260
Columns with finite uzal_cost: 75


## Визуализация качества вложения по критерию Узала

In [ ]:
# ошибки реконструкции
# ошибки обусловленные шумами

def plot_random_uzal_quality(
    results_df,
    n_series=15,
    random_state=42,
    min_points=5
):
    """
    Строит графики зависимости качества вложения Uzal
    от лага tau и размерности dimension для n_series случайных рядов.

    На каждом графике крестиком отмечается лучший результат
    (минимальное значение uzal_cost).
    """

    required_columns = {"column", "tau", "dimension", "uzal_cost"}
    missing_columns = required_columns - set(results_df.columns)

    if missing_columns:
        raise ValueError(f"В таблице нет колонок: {sorted(missing_columns)}")

    df = results_df.dropna(subset=["column", "tau", "dimension", "uzal_cost"]).copy()

    df["tau"] = df["tau"].astype(int)
    df["dimension"] = df["dimension"].astype(int)

    # Оставляем только те ряды, где достаточно точек
    counts = df.groupby("column").size()
    valid_columns = counts[counts >= min_points].index.to_numpy()

    if len(valid_columns) == 0:
        raise ValueError("Нет рядов с достаточным количеством значений uzal_cost.")

    n_series = min(n_series, len(valid_columns))

    rng = np.random.default_rng(random_state)
    selected_columns = rng.choice(valid_columns, size=n_series, replace=False)

    print("Выбранные ряды:")
    for col in selected_columns:
        print("-", col)

    for col in selected_columns:
        one = df[df["column"] == col].copy()

        # Лучшая строка = минимальный uzal_cost
        best_row = one.loc[one["uzal_cost"].idxmin()]
        best_tau = best_row["tau"]
        best_dim = best_row["dimension"]
        best_cost = best_row["uzal_cost"]

        # Таблица: строки — dimension, столбцы — tau, значения — uzal_cost
        pivot = one.pivot_table(
            index="dimension",
            columns="tau",
            values="uzal_cost",
            aggfunc="mean"
        )

        fig = px.imshow(
            pivot,
            labels={
                "x": "Лаг tau",
                "y": "Размерность embedding",
                "color": "Uzal cost"
            },
            title=f"Качество embedding по Uzal для ряда: {col}",
            aspect="auto"
        )

        # Добавляем крестик в точку лучшего результата
        fig.add_scatter(
            x=[best_tau],
            y=[best_dim],
            mode="markers+text",
            marker=dict(
                symbol="x",
                size=14,
                color="cyan",
                line=dict(width=2)
            ),
            text=[f"best<br>tau={best_tau}, dim={best_dim}<br>{best_cost:.3f}"],
            textposition="top center",
            name="Лучший результат"
        )

        fig.update_layout(
            width=850,
            height=550
        )

        fig.show()

        print(
            f"Лучший результат для {col}: "
            f"tau={best_tau}, dimension={best_dim}, uzal_cost={best_cost:.6f}"
        )

plot_random_uzal_quality(finite_results, n_series=15, random_state=42)

Выбранные ряды:
- Lab1_G1_T1002
- Lab1_G1_T1003
- Lab1_G3_Pc3
- Lab1_dev
- Lab1_G2_2F3
- Lab1_G4_T4пр
- Lab1_Rc
- Lab1_G4_T4
- Lab1_G2_Fн9
- Lab1_G4_delta_T4
- Lab1_PposleNag
- Lab1_G3_Lm
- Lab1_G3_V1
- Lab1_G3_T638
- Lab1_TC_Tm


Лучший результат для Lab1_G1_T1002: tau=27, dimension=6, uzal_cost=-0.799557


Лучший результат для Lab1_G1_T1003: tau=100, dimension=3, uzal_cost=-0.351305


Лучший результат для Lab1_G3_Pc3: tau=34, dimension=3, uzal_cost=-1.477546


Лучший результат для Lab1_dev: tau=9, dimension=4, uzal_cost=-2.216803


...

Лучший результат для Lab1_PposleNag: tau=33, dimension=4, uzal_cost=-1.782888


Лучший результат для Lab1_G3_Lm: tau=32, dimension=3, uzal_cost=-3.309205


Лучший результат для Lab1_G3_V1: tau=15, dimension=3, uzal_cost=-0.961995


Лучший результат для Lab1_G3_T638: tau=25, dimension=3, uzal_cost=-1.935006


Лучший результат для Lab1_TC_Tm: tau=83, dimension=10, uzal_cost=-0.939020


In [26]:
best_idx = finite_results.groupby("column")["uzal_cost"].idxmin()

best_choices = (
    finite_results.loc[best_idx, ["column", "tau", "dimension", "uzal_cost"]]
    .sort_values("uzal_cost")
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)

display(best_choices)

,column,tau,dimension,uzal_cost
0,Lab1_G3_Lm,32,3,-3.309205
1,Lab1_Rc,80,11,-2.887453
2,Lab1_Hpol,83,5,-2.498611
3,Lab1_he,11,11,-2.350634
4,Lab1_dev,9,4,-2.216803
5,Lab1_G3_T638,25,3,-1.935006
6,Lab1_G3_T600,52,3,-1.913061
7,Lab1_PdoNag,79,5,-1.828615
8,Lab1_dPmg,76,9,-1.808239
9,Lab1_Pm_sm_N,95,10,-1.803679


In [24]:
from pathlib import Path

paths = [
    Path("uzal_columns_without_finite_cost.csv"),
    Path("uzal_embedding_results_all_columns.csv"),
    Path("uzal_embedding_results.csv"),
    Path("uzal_embedding_skipped_columns.csv"),
    Path("uzal_best_choices_by_column.csv")
]
  
for path in paths:
    if path.exists():
        path.unlink()

best_choices.to_csv("uzal_cost_final_result.csv")